# Exploration Alibaba Cluster Trace 2017
Le fichier `batch_task.csv` contient **8 colonnes**, sans en-tete.
Schema officiel : https://github.com/alibaba/clusterdata/blob/master/cluster-trace-v2017/schema.csv

Les demandes CPU/RAM absentes restent manquantes. La memoire est normalisee, pas en Go.
Les valeurs CPU sont conservees sans conversion. `create_timestamp` est une date de creation,
pas de debut d'execution. Le deuxieme champ est nomme fin dans schema.csv et derniere
modification d'etat dans trace_201708.md : ne pas en deduire une duree d'execution.
Les timestamps negatifs sont conserves. `batch_instance.csv` n'est pas charge ici.


In [ ]:
from pathlib import Path
import sys

# Works from the project root or ML/notebooks.
project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "ML" / "src" / "preprocess.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Open this notebook from the ai-cloud-optimizer project.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ML.src.preprocess import BATCH_TASK_COLUMNS, load_batch_tasks

In [ ]:
# Alibaba 2017: 8 columns, no task_type or task_name.
batch_task_columns = list(BATCH_TASK_COLUMNS)
batch_task_columns

In [ ]:
batch_task_path = project_root / "ML" / "data" / "raw" / "batch_task.csv"
batch_task = load_batch_tasks(batch_task_path)

In [ ]:
batch_task.head()

In [ ]:
batch_task.shape

In [ ]:
batch_task.info()

In [ ]:
batch_task.describe()

In [ ]:
batch_task.isnull().sum()

In [ ]:
batch_task["status"].value_counts()

In [ ]:
# Number of instances per task; task_type does not exist in v2017.
batch_task["instance_num"].describe()

In [ ]:
# Preserve and inspect missing resource requests by status.
batch_task.assign(
    missing_plan_cpu=batch_task["plan_cpu"].isna(),
    missing_plan_mem=batch_task["plan_mem"].isna(),
).groupby("status")[["missing_plan_cpu", "missing_plan_mem"]].sum()